# Laboratorio 01.1 — Tokens y ventana de contexto

Construiremos un tokenizador BPE didáctico pequeño y veremos cómo un presupuesto de contexto limita la entrada disponible para una respuesta.

## 1. Objetivos de aprendizaje

Al finalizar podrás:
- Explicar por qué los modelos procesan tokens y no palabras completas.
- Implementar fusiones BPE sencillas a partir de un corpus pequeño.
- Comparar recuentos de palabras y tokens sin confundir la aproximación con un tokenizer comercial.
- Calcular cuánto contexto queda después de reservar espacio para la respuesta.

## 2. Conceptos

Un token puede ser una palabra, parte de una palabra, puntuación o bytes agrupados. BPE comienza con unidades pequeñas y fusiona pares frecuentes; el vocabulario y el texto alteran el número de tokens. La ventana de contexto contiene la entrada, el historial y la salida generada. Si el límite es $C$, el espacio de entrada debe reservar lugar para la respuesta: $C_{entrada} \leq C - C_{salida}$. El algoritmo de este laboratorio es solo una demostración y no reproduce el tokenizador de un proveedor.

## 3. Arquitectura

```mermaid
flowchart LR
    A[Corpus pequeno] --> B[Frecuencias de pares]
    B --> C[Fusiones BPE]
    D[Texto de entrada] --> E[Tokenizador de juguete]
    C --> E
    E --> F[Recuento de tokens]
    F --> G[Presupuesto de contexto]
```

## 4. Preparación del entorno

Importamos solo expresiones regulares, contadores y bibliotecas de análisis. El inicializador compartido carga `.env` sin mostrar secretos; aquí no se realizan solicitudes a modelos.

In [ ]:
from collections import Counter
import re

import matplotlib.pyplot as plt
import pandas as pd

from llm_engineering.environment import preparar_entorno

raiz_curso = preparar_entorno()

## 5. Definición del problema

Estimaremos el tamaño relativo de instrucciones en español y evaluaremos si una petición cabe en una ventana de contexto hipotética. El corpus de entrenamiento BPE está escrito para este laboratorio; no es un dataset de evaluación de ningún proveedor.

## 6. Línea base

La línea base cuenta palabras separadas por espacios. Es barata y fácil de explicar, pero subcuenta puntuación y fragmentos de palabras. Primero fijamos un texto reproducible para comparar los métodos.

In [ ]:
corpus = [
    'los modelos predicen texto',
    'los modelos usan contexto',
    'el contexto limita la respuesta',
    'predecir tokens requiere datos',
    'los tokens representan texto',
]
texto_prueba = 'Los modelos de lenguaje predicen tokens y deben respetar el presupuesto de contexto.'
recuento_palabras = len(texto_prueba.split())
print(f'Palabras en la línea base: {recuento_palabras}')

## 7. Implementación

Representamos cada palabra como una secuencia de caracteres, contamos pares adyacentes y fusionamos el par más frecuente. El desempate lexicográfico hace determinista este pequeño experimento.

In [ ]:
def fusionar_par(secuencia: tuple[str, ...], par: tuple[str, str]) -> tuple[str, ...]:
    """Fusiona todas las apariciones no solapadas de un par adyacente."""
    resultado = []
    indice = 0
    while indice < len(secuencia):
        if indice + 1 < len(secuencia) and secuencia[indice:indice + 2] == par:
            resultado.append(secuencia[indice] + secuencia[indice + 1])
            indice += 2
        else:
            resultado.append(secuencia[indice])
            indice += 1
    return tuple(resultado)


def aprender_bpe(textos: list[str], cantidad_fusiones: int) -> list[tuple[str, str]]:
    """Aprende fusiones frecuentes sobre palabras de un corpus pequeno."""
    vocabulario = Counter(
        tuple(palabra)
        for texto in textos
        for palabra in re.findall(r'\w+', texto.lower(), flags=re.UNICODE)
    )
    fusiones = []
    for _ in range(cantidad_fusiones):
        frecuencias = Counter()
        for secuencia, frecuencia in vocabulario.items():
            frecuencias.update({par: frecuencia for par in zip(secuencia, secuencia[1:])})
        if not frecuencias:
            break
        par = min(frecuencias, key=lambda candidato: (-frecuencias[candidato], candidato))
        fusiones.append(par)
        nuevo_vocabulario = Counter()
        for secuencia, frecuencia in vocabulario.items():
            nuevo_vocabulario[fusionar_par(secuencia, par)] += frecuencia
        vocabulario = nuevo_vocabulario
    return fusiones


def tokenizar_juguete(texto: str, fusiones: list[tuple[str, str]]) -> list[str]:
    """Tokeniza palabras y puntuacion aplicando las fusiones aprendidas."""
    tokens = []
    for pieza in re.findall(r'\w+|[^\w\s]', texto.lower(), flags=re.UNICODE):
        if re.fullmatch(r'\w+', pieza, flags=re.UNICODE):
            secuencia = tuple(pieza)
            for par in fusiones:
                secuencia = fusionar_par(secuencia, par)
            tokens.extend(secuencia)
        else:
            tokens.append(pieza)
    return tokens

fusiones = aprender_bpe(corpus, cantidad_fusiones=12)
tokens = tokenizar_juguete(texto_prueba, fusiones)
print('Fusiones aprendidas:', fusiones)
print('Tokens de juguete:', tokens)

## 8. Experimentos

Cambiamos solo la cantidad de fusiones y mantenemos fijo el texto y corpus. Más fusiones pueden compactar secuencias frecuentes, pero no garantizan menos tokens para cualquier texto.

In [ ]:
recuentos = []
for cantidad in (0, 4, 8, 12, 20):
    fusiones_experimento = aprender_bpe(corpus, cantidad)
    recuentos.append({
        'fusiones': cantidad,
        'palabras': recuento_palabras,
        'tokens_juguete': len(tokenizar_juguete(texto_prueba, fusiones_experimento)),
    })
tabla_tokens = pd.DataFrame(recuentos)
display(tabla_tokens)
tabla_tokens.to_csv(raiz_curso / 'outputs' / '01_01_token_counts.csv', index=False)
tabla_tokens.plot(x='fusiones', y=['palabras', 'tokens_juguete'], marker='o', ylabel='Unidades contadas')
plt.show()

## 9. Evaluación

Usamos el tokenizador de juguete para estimar si una entrada cabe después de reservar tokens de salida. La métrica es una comprobación de presupuesto, no una garantía de que un modelo real use el mismo conteo.

In [ ]:
limite_contexto = 36
reserva_salida = 12
presupuesto_entrada = limite_contexto - reserva_salida
texto_entrada = texto_prueba + ' ' + 'Resume los hallazgos y cita cada afirmacion.'
tokens_entrada = len(tokenizar_juguete(texto_entrada, fusiones))
evaluacion_contexto = pd.DataFrame([{
    'limite_total': limite_contexto,
    'reserva_salida': reserva_salida,
    'presupuesto_entrada': presupuesto_entrada,
    'tokens_entrada_estimados': tokens_entrada,
    'cabe': tokens_entrada <= presupuesto_entrada,
}])
display(evaluacion_contexto)
evaluacion_contexto.to_csv(raiz_curso / 'outputs' / '01_01_context_budget.csv', index=False)

## 10. Discusión

El conteo por espacios es una línea base, no un estimador de tokens fiable. BPE explica por qué fragmentos recurrentes pueden fusionarse, pero los tokenizadores reales tienen vocabularios, reglas y codificaciones de bytes distintos. También hay que presupuestar mensajes del sistema, historial, herramientas y la respuesta.

## 11. Ejercicios

1. Añade nombres propios y palabras con tildes al corpus; compara su fragmentación.
2. Cambia solo el número de fusiones y explica los cambios observados.
3. Modifica el presupuesto de salida y encuentra una petición que deje de caber.

## 12. Desafío

Amplía la cuenta a una conversación con varios turnos, encabezados de rol y una respuesta reservada. Diseña una tabla que muestre qué componente consume cada parte del presupuesto.

## 13. Ideas clave

- Los tokens no equivalen a palabras.
- El corpus y el vocabulario de fusiones cambian el conteo.
- Una ventana de contexto debe reservar espacio para la salida.
- Un tokenizador de juguete enseña el mecanismo, pero no sustituye al del modelo elegido.